<a href="https://colab.research.google.com/github/ihdyrtg/data_science/blob/main/colab_klasifikasi_risiko_siber_pemda.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Google Colab: Klasifikasi Risiko Request Siber Berbasis Machine Learning

**Tujuan praktikum/riset:** membangun model machine learning untuk mengklasifikasikan status request keamanan web: `label=0` berarti request diizinkan/HTTP 200, sedangkan `label=1` berarti request diblokir/HTTP 403.

**Catatan metodologis penting:** kolom `response_code` tidak dipakai sebagai fitur karena nilainya identik dengan label. Memakai kolom ini sama seperti mahasiswa melihat kunci jawaban sebelum ujian; skor model akan tampak sempurna tetapi tidak valid secara ilmiah.

**Rancangan penelitian yang disarankan:** *Explainable Machine Learning for Risk-Based Classification of Malicious Web Requests in Local Government Web Security Logs*.

## 1. Setup Library
Jalankan cell ini terlebih dahulu. Notebook memakai scikit-learn agar mudah dijalankan di Google Colab tanpa instalasi tambahan.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, average_precision_score, confusion_matrix,
                             classification_report, ConfusionMatrixDisplay)

RANDOM_STATE = 42

## 2. Upload dan Baca Dataset
Unggah file CSV dataset ke Colab. Bila file sudah berada di `/content/`, sesuaikan nama file pada variabel `file_path`.

In [ ]:
try:
    from google.colab import files
    uploaded = files.upload()
    if uploaded:
        file_path = list(uploaded.keys())[0]
    else:
        file_path = 'Dataset_risiko_siber.xlsx'
except Exception:
    file_path = '/content/Dataset_risiko_siber.xlsx'

df = pd.read_excel(file_path)
print(df.shape)
df.head()

(14951, 12)


,request_per_minute,payload_size_bytes,response_code,url_length,param_count,has_sql_keyword,has_xss_pattern,hour_of_day,session_duration_sec,failed_login_count,is_known_bad_ip,attack_type
0,2,15,0,74,2,0,0,7,82829,1,0,Code Inj
1,4,31,0,29,0,0,0,7,4700304,3,0,Cmd Inj
2,3,15,0,66,2,0,0,7,226192,2,0,Code Inj
3,1,328,0,34,0,0,0,6,0,0,0,Unserialize
4,1,1465,0,13,0,0,0,6,0,0,0,Unserialize


## 3. Audit Kualitas Data
Bagian ini wajib untuk artikel Sinta 1/2. Reviewer akan melihat apakah data layak dipakai, apakah ada missing value, fitur konstan, dan potensi data leakage.

In [ ]:
print('Shape:', df.shape)
print('Kolom:', df.columns.tolist())
print('\nMissing values:')
print(df.isna().sum())
print('\nJumlah nilai unik per kolom:')
print(df.nunique(dropna=False).sort_values())
print('\nDistribusi label:')
print(df['attack_type'].value_counts())
print((df['attack_type'].value_counts(normalize=True)*100).round(2))
print('\nCrosstab response_code vs label:')
print(pd.crosstab(df['response_code'], df['attack_type']))

Shape: (14951, 12)
Kolom: ['request_per_minute', 'payload_size_bytes', 'response_code', 'url_length', 'param_count', 'has_sql_keyword', 'has_xss_pattern', 'hour_of_day', 'session_duration_sec', 'failed_login_count', 'is_known_bad_ip', 'attack_type']

Missing values:
request_per_minute      0
payload_size_bytes      0
response_code           0
url_length              0
param_count             0
has_sql_keyword         0
has_xss_pattern         0
hour_of_day             0
session_duration_sec    0
failed_login_count      0
is_known_bad_ip         0
attack_type             0
dtype: int64

Jumlah nilai unik per kolom:
response_code              2
has_sql_keyword            2
has_xss_pattern            2
is_known_bad_ip            2
param_count                8
attack_type               10
hour_of_day               24
failed_login_count        45
request_per_minute       104
url_length               174
payload_size_bytes       279
session_duration_sec    1180
dtype: int64

Distribusi label

### Interpretasi audit awal
- `response_code` harus dikeluarkan dari fitur karena 200 selalu label 0 dan 403 selalu label 1.
- `bytes_sent` kosong seluruhnya, sehingga dihapus.
- `request_method_enc`, `user_agent_suspicious`, dan `redirect_count` konstan, sehingga tidak membantu model.
- `attack_type` sebaiknya tidak dipakai pada model utama jika fitur tersebut berasal dari hasil klasifikasi WAF, karena bisa menjadi informasi pasca-deteksi. Gunakan hanya untuk analisis deskriptif.

## 4. Eksplorasi Data

In [ ]:
ax = df['label'].value_counts().sort_index().plot(kind='bar')
ax.set_title('Distribusi Label')
ax.set_xlabel('Label')
ax.set_ylabel('Jumlah Request')
plt.show()

ax = df['attack_type'].value_counts().sort_values().plot(kind='barh', figsize=(8,5))
ax.set_title('Distribusi Jenis Serangan')
ax.set_xlabel('Jumlah Request')
plt.show()

print(pd.crosstab(df['attack_type'], df['label']))

## 5. Menyiapkan Fitur yang Aman
Fitur yang dikeluarkan: `response_code` karena leakage; `attack_type` karena kemungkinan berasal dari sistem deteksi; `bytes_sent` karena kosong; dan fitur konstan karena tidak informatif.

In [ ]:
target = 'label'
leakage_cols = ['response_code']
constant_or_empty_cols = ['bytes_sent', 'request_method_enc', 'user_agent_suspicious', 'redirect_count']
post_detection_cols = ['attack_type']

drop_cols = [target] + leakage_cols + constant_or_empty_cols + post_detection_cols
X = df.drop(columns=drop_cols)
y = df[target]

print('Fitur final:', X.columns.tolist())
print('Shape X:', X.shape)

## 6. Split Data dan Pipeline Preprocessing
Preprocessing di-fit hanya pada data latih. Ini mencegah kebocoran informasi dari data uji ke proses pelatihan.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

num_cols = X.columns.tolist()
preprocess = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
    ]), num_cols)
])

## 7. Pelatihan Model
Model yang dibandingkan: Logistic Regression sebagai baseline, Random Forest, Gradient Boosting, dan SVM-RBF.

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced'),
    'Random Forest': RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE,
                                            class_weight='balanced_subsample', min_samples_leaf=2),
    'Gradient Boosting': GradientBoostingClassifier(random_state=RANDOM_STATE),
    'SVM-RBF': SVC(kernel='rbf', probability=True, class_weight='balanced', random_state=RANDOM_STATE)
}

results = []
fitted_models = {}
for name, model in models.items():
    pipe = Pipeline([('preprocess', preprocess), ('model', model)])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    proba = pipe.predict_proba(X_test)[:, 1]
    cm = confusion_matrix(y_test, pred)
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, pred),
        'Precision': precision_score(y_test, pred),
        'Recall': recall_score(y_test, pred),
        'F1-Score': f1_score(y_test, pred),
        'ROC-AUC': roc_auc_score(y_test, proba),
        'PR-AUC': average_precision_score(y_test, proba),
        'TN': cm[0,0], 'FP': cm[0,1], 'FN': cm[1,0], 'TP': cm[1,1]
    })
    fitted_models[name] = pipe

result_df = pd.DataFrame(results).sort_values('F1-Score', ascending=False)
result_df

## 8. Evaluasi Model Terbaik

In [ ]:
best_name = result_df.iloc[0]['Model']
best_model = fitted_models[best_name]
pred = best_model.predict(X_test)
print('Model terbaik:', best_name)
print(classification_report(y_test, pred, digits=4))

ConfusionMatrixDisplay.from_predictions(y_test, pred, display_labels=['Allowed', 'Blocked'])
plt.title(f'Confusion Matrix - {best_name}')
plt.show()

## 9. Validasi Silang 5-Fold
Validasi silang membantu memastikan performa tidak hanya kebetulan muncul pada satu pembagian data.

In [ ]:
cv_models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced'),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE,
                                            class_weight='balanced_subsample', min_samples_leaf=2),
    'Gradient Boosting': GradientBoostingClassifier(random_state=RANDOM_STATE)
}
scoring = {'accuracy':'accuracy','precision':'precision','recall':'recall','f1':'f1','roc_auc':'roc_auc','pr_auc':'average_precision'}
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_rows = []
for name, model in cv_models.items():
    pipe = Pipeline([('preprocess', preprocess), ('model', model)])
    cv = cross_validate(pipe, X, y, cv=skf, scoring=scoring)
    row = {'Model': name}
    for metric in scoring:
        row[metric.upper() + '_Mean'] = cv['test_' + metric].mean()
        row[metric.upper() + '_SD'] = cv['test_' + metric].std()
    cv_rows.append(row)

cv_result_df = pd.DataFrame(cv_rows)
cv_result_df

## 10. Interpretasi Fitur Penting
Feature importance membantu menjawab: fitur apa yang paling banyak dipakai model untuk membedakan request blocked dan allowed?

In [ ]:
rf_model = fitted_models['Random Forest'].named_steps['model']
fi = pd.DataFrame({
    'Feature': num_cols,
    'Importance': rf_model.feature_importances_
}).sort_values('Importance', ascending=False)
fi

ax = fi.sort_values('Importance').plot(x='Feature', y='Importance', kind='barh', legend=False, figsize=(8,5))
ax.set_title('Feature Importance - Random Forest')
plt.show()

## 11. Eksperimen Tambahan: Memasukkan `attack_type` sebagai Fitur
Eksperimen ini hanya untuk analisis tambahan. Jangan jadikan model utama jika `attack_type` adalah hasil dari mesin deteksi/WAF, karena itu termasuk informasi pasca-deteksi.

In [ ]:
X2 = df.drop(columns=['label','response_code','bytes_sent','request_method_enc','user_agent_suspicious','redirect_count'])
y2 = df['label']
num_cols2 = [c for c in X2.columns if c != 'attack_type']
cat_cols2 = ['attack_type']
preprocess2 = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num_cols2),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols2)
])
X_train2, X_test2, y_train2, y_test2 = train_test_split(X2, y2, test_size=0.2, stratify=y2, random_state=RANDOM_STATE)
rf2 = Pipeline([('preprocess', preprocess2), ('model', RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, class_weight='balanced_subsample', min_samples_leaf=2))])
rf2.fit(X_train2, y_train2)
pred2 = rf2.predict(X_test2)
proba2 = rf2.predict_proba(X_test2)[:,1]
print(classification_report(y_test2, pred2, digits=4))
print('ROC-AUC:', roc_auc_score(y_test2, proba2))

## 12. Simpan Hasil
File hasil dapat dipakai untuk tabel artikel.

In [ ]:
result_df.to_csv('model_holdout_results.csv', index=False)
cv_result_df.to_csv('model_crossval_results.csv', index=False)
fi.to_csv('rf_feature_importance.csv', index=False)
print('Hasil disimpan sebagai CSV.')

## 13. Refleksi untuk Artikel
Jawab pertanyaan berikut pada bagian pembahasan artikel:
1. Mengapa `response_code` harus dikeluarkan dari fitur?
2. Mengapa Random Forest lebih kuat daripada Logistic Regression pada dataset ini?
3. Apakah model ini memprediksi kerentanan situs? Jawaban ketat: belum. Model ini memprediksi status request berdasarkan pola perilaku log. Validasi kerentanan memerlukan VA terkontrol.
4. Variabel tambahan apa yang diperlukan agar penelitian naik kelas menjadi prediksi endpoint rentan?